[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.3 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        self.W_q = nn.Linear(d_model, d_model) # demb
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.num_heads = num_heads
        self.d_model = d_model
        self.dk = self.d_model // self.num_heads # dim per head

    def forward(self, Q, K, V):
        bs = Q.size(0) # batch size
        sq = Q.size(1) # q sequence len
        sk = K.size(1) # kv seq len
        Q = self.W_q(Q).reshape(bs, sq, self.num_heads, self.dk) # bs, sq, demb or bs, sq, nh, dk
        K = self.W_k(K).reshape(bs, sk, self.num_heads, self.dk)
        V = self.W_v(V).reshape(bs, sk, self.num_heads, self.dk)
        Q = torch.transpose(Q, 1, 2)# bs, nh, sq, dk 
        K = torch.transpose(K, 1, 2)# bs, nh, sk, dk
        V = torch.transpose(V, 1, 2) 
        # print(Q @ K.mT) # BS, nh, sq, sk
        attn = torch.softmax(Q @ K.mT / math.sqrt(self.dk), -1) @ V # BS, nh, sq, sk
        attn = torch.transpose(attn, 1, 2) # bs, sq, nh, dk
        attn = attn.reshape(bs, sq, self.d_model)
        return self.W_o(attn)


In [76]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
# y = torch.randn(2, 8, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
Output shape: torch.Size([2, 6, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [77]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/6] Output shape (5.0ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (1.1ms)
  ✅ [3/6] Numerical correctness vs reference (5.3ms)
  ✅ [4/6] Gradient flow (4.4ms)
  ✅ [5/6] Cross-attention (seq_q != seq_k) (1.4ms)
  ✅ [6/6] Different heads give different outputs (2.7ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (19.9ms total)
  Progress saved. Run status() to see your dashboard.

